In [1]:
from google.colab import drive
MAIN_FOLDER="/content/drive/MyDrive/לימודים תואר שני/Year_2/semester_2/LLmCourse/ex1/code-and-data"
DATA_FOLDER=MAIN_FOLDER+"/data"
CODE_FOLDER=MAIN_FOLDER+"/code"
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import sys
# Append the folder path to sys.path so Python can find the local modules
if MAIN_FOLDER not in sys.path:
    sys.path.append(MAIN_FOLDER)
    print(f'Added {MAIN_FOLDER} to sys.path')
if DATA_FOLDER not in sys.path:
    sys.path.append(DATA_FOLDER)
    print(f'Added {DATA_FOLDER} to sys.path')
if CODE_FOLDER not in sys.path:
    sys.path.append(CODE_FOLDER)
    print(f'Added {CODE_FOLDER} to sys.path')

Added /content/drive/MyDrive/לימודים תואר שני/Year_2/semester_2/LLmCourse/ex1/code-and-data to sys.path
Added /content/drive/MyDrive/לימודים תואר שני/Year_2/semester_2/LLmCourse/ex1/code-and-data/data to sys.path
Added /content/drive/MyDrive/לימודים תואר שני/Year_2/semester_2/LLmCourse/ex1/code-and-data/code to sys.path


In [3]:
from __future__ import annotations

import torch
from visualize import extract_and_plot
import lm
import torch
from torch import nn, optim
from transformer import TransformerLM
import data
!pip install optuna

import optuna
import torch
import torch.optim as optim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 8.5 MB/s eta 0:00:00


In [ ]:
model: torch.nn.Module = TransformerLM(
        n_layers,
        n_heads,
        embed_size,
        seq_len,
        tokenizer.vocab_size(),
        mlp_hidden_size,
        with_residuals=True,
    )

model = model.to(device)
optimizer = optim.AdamW(model.parameters(), lr=learning_rate, betas=[0.9, 0.95])
num_batches = 0

Parameter count: 2.51M


In [4]:
import os

# Define the paths for the directories
models_heb_dir = os.path.join(MAIN_FOLDER, 'models_heb') # Use MAIN_FOLDER for relative path consistency
figs_heb_dir = os.path.join(MAIN_FOLDER, 'figs_heb')
models_eng_dir = os.path.join(MAIN_FOLDER, 'models_eng') # Use MAIN_FOLDER for relative path consistency
figs_dir = os.path.join(MAIN_FOLDER, 'figs_eng')
figs_final_dir=os.path.join(MAIN_FOLDER, 'figs_final')
models_final_dir = os.path.join(MAIN_FOLDER, 'models_final')
# Create the directories if they don't exist

# Check if the 'models' directory exists, and create it if not
if not os.path.exists(models_heb_dir):
    os.makedirs(models_heb_dir)
    print(f"Created directory: {models_heb_dir}")
else:
    print(f"Directory already exists: {models_heb_dir}")

if not os.path.exists(models_eng_dir):
    os.makedirs(models_eng_dir)
    print(f"Created directory: {models_eng_dir}")
else:
    print(f"Directory already exists: {models_eng_dir}")

if not os.path.exists(models_final_dir):
    os.makedirs(models_final_dir)
    print(f"Created directory: {models_final_dir}")
else:
    print(f"Directory already exists: {models_final_dir}")
# Check if the 'figs' directory exists, and create it if not
if not os.path.exists(figs_dir):
    os.makedirs(figs_dir)
    print(f"Created directory: {figs_dir}")
else:
    print(f"Directory already exists: {figs_dir}")

if not os.path.exists(figs_final_dir):
    os.makedirs(figs_final_dir)
    print(f"Created directory: {figs_final_dir}")
else:
    print(f"Directory already exists: {figs_final_dir}")


Directory already exists: /content/drive/MyDrive/לימודים תואר שני/Year_2/semester_2/LLmCourse/ex1/code-and-data/models_heb
Directory already exists: /content/drive/MyDrive/לימודים תואר שני/Year_2/semester_2/LLmCourse/ex1/code-and-data/models_eng
Directory already exists: /content/drive/MyDrive/לימודים תואר שני/Year_2/semester_2/LLmCourse/ex1/code-and-data/models_final
Directory already exists: /content/drive/MyDrive/לימודים תואר שני/Year_2/semester_2/LLmCourse/ex1/code-and-data/figs_eng
Directory already exists: /content/drive/MyDrive/לימודים תואר שני/Year_2/semester_2/LLmCourse/ex1/code-and-data/figs_final


In [5]:

from utils import save_best_model,parameters,loss_plotter,split_data
def objective(trial):
    # 1. Define the hyperparameter search space
    seq_len = trial.suggest_categorical("seq_len", [128, 256])
    batch_size = trial.suggest_categorical("batch_size", [32, 64, 128])
    n_layers = trial.suggest_categorical("n_layers", [4, 6, 8])
    n_heads = trial.suggest_categorical("n_heads", [4,8])
    embed_size = trial.suggest_categorical("embed_size", [128, 256, 384])
    dropouts=trial.suggest_categorical("dropouts", [None,0.1, 0.2])

    # Assuming learning rate should be a log-uniform float. Adjust bounds as needed.
    learning_rate = trial.suggest_float("learning_rate", 1e-5, 1e-2, log=True)

    mlp_hidden_size = embed_size * 4

    params = {
        "seq_len": seq_len,
        "batch_size": batch_size,
        "n_layers": n_layers,
        "n_heads": n_heads,
        "embed_size": embed_size,
        "mlp_hidden_size": mlp_hidden_size,
        "learning_rate": learning_rate,
        "dropout_rate": dropouts
    }

    # 2. Data Split and Iterators
    train_data, val_data = split_data(tokenized_data, seq_len)

    train_iter = iter(data.RandomOrderDataIterator(train_data, seq_len + 1))
    val_iter = iter(data.RandomOrderDataIterator(val_data, seq_len + 1))

    print(f'Training model with params: {params}')
    # 3. Initialize Model
    model: torch.nn.Module = TransformerLM(
        params['n_layers'],
        params['n_heads'],
        params['embed_size'],
        params['seq_len'],
        tokenizer.vocab_size(),
        params['mlp_hidden_size'],
        with_residuals=True,
        dropout=[params["dropout_rate"],params["dropout_rate"],params["dropout_rate"]]
    )
    model = model.to(device)

    # 4. Setup Optimizer & Scheduler
    optimizer = optim.AdamW(model.parameters(), lr=learning_rate, betas=[0.9, 0.95])
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode='min',
        factor=0.5,
        patience=5,
        threshold=1e-4,
        min_lr=1e-7
    )

    # 5. Training Loop Variables
    best_val_loss = float('inf')
    val_losses = []
    train_losses = []
    num_batches = 0
    early_stopping_patience = 7# Number of validation checks to wait before stopping
    epochs_no_improve = 0
    model.train()

    for batch in data.batch_items(train_iter, batch_size):
        if num_batches >= num_batches_to_train:
            break

        batch_x, batch_y = lm.batch_to_labeled_samples(batch)
        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)

        logits = model(batch_x)
        loss = lm.compute_loss(logits, batch_y)

        # Parameter update
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), gradient_clipping)
        optimizer.step()

        num_batches += 1

        if num_batches % 10 == 0:
            # print(f"Seen {num_batches} batches. last loss is: {loss.item()}")
            train_losses.append(loss.item())

            # if num_batches % 100 == 0:
                # for _ in range(1):
                    # model.eval()
                    # sampled = tokenizer.detokenize(
                        # model.sample_continuation(tokenizer.tokenize("Hello"), 500)
                    # )
                    # model.train()
                    # print(f"Model sample: '''{sampled}'''")
                # print("")

        if num_batches % 100 == 0:
            model.eval()
            with torch.no_grad():
                val_batch = None
                try:
                    val_batch = next(data.batch_items(val_iter, batch_size))
                except StopIteration:
                    val_iter = iter(data.RandomOrderDataIterator(val_data, seq_len + 1))
                    val_batch = next(data.batch_items(val_iter, batch_size))

                val_x, val_y = lm.batch_to_labeled_samples(val_batch)
                val_x = val_x.to(device)
                val_y = val_y.to(device)

                val_logits = model(val_x)
                val_loss = lm.compute_loss(val_logits, val_y)
                curr_val_loss = val_loss.item()
                val_losses.append(curr_val_loss)
                scheduler.step(curr_val_loss)

                if curr_val_loss < best_val_loss:
                    best_val_loss = curr_val_loss
                    epochs_no_improve = 0  # Reset patience counter
                    # save_best_model(model, params, best_val_loss, epoch=num_batches, out_dir=models_dir)
                else:
                    epochs_no_improve += 1 # Increment patience counter
                # print(f"Val loss: {curr_val_loss} best val loss: {best_val_loss}")

            # --- OPTUNA PRUNING CHECK ---
            trial.report(curr_val_loss, num_batches)
            if trial.should_prune():
                raise optuna.exceptions.TrialPruned()
            # ---  TRADITIONAL EARLY STOPPING CHECK ---
            if epochs_no_improve >= early_stopping_patience:
                print("Early stopping triggered: Validation loss stagnated.")
                break # Breaks out of the training loop, ending this trial early
            model.train()
        if num_batches %500 ==0:
          print(f"batch: {num_batches} Val loss: {curr_val_loss} best val loss: {best_val_loss}")


    # Plot metrics at the end of the trial
    loss_plotter(train_losses, val_losses, params, figs_dir)

    # 6. Return the objective value for Optuna to minimize
    return best_val_loss

In [16]:
print(torch.cuda.is_available())
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
data_path = DATA_FOLDER+"/he/"
gradient_clipping = 1.0
num_batches_to_train = 6000
tokenizer, tokenized_data = data.load_data(data_path)
# NOTE: are data items are longer by one than the sequence length,
# They will be shortened by 1 when converted to training examples.
# data_iter = iter(data.RandomOrderDataIterator(tokenized_data, seq_len + 1))

True
Using device: cuda:0


In [18]:
# Create study and optionally use a pruner (MedianPruner is standard)
study = optuna.create_study(
    direction="minimize",
    pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=500)
)

# Run for a specified number of trials
study.optimize(objective, n_trials=40)

# Output the best results
print("\nOptimization Finished!")
print("Best trial:")
best_trial = study.best_trial

print(f"  Value (Best Val Loss): {best_trial.value}")
print("  Best Params: ")
for key, value in best_trial.params.items():
    print(f"    {key}: {value}")

[I 2026-04-20 17:34:43,360] A new study created in memory with name: no-name-44680a30-fb32-42eb-9d4d-f9b7c022f748


Training model with params: {'seq_len': 128, 'batch_size': 32, 'n_layers': 4, 'n_heads': 4, 'embed_size': 384, 'mlp_hidden_size': 1536, 'learning_rate': 0.0036166956805212896, 'dropout_rate': 0.1}
Parameter count: 6.62M
batch: 500 Val loss: 2.661163806915283 best val loss: 2.643864870071411
batch: 1000 Val loss: 2.5949409008026123 best val loss: 2.5949409008026123
batch: 1500 Val loss: 2.6237454414367676 best val loss: 2.4834513664245605
Early stopping triggered: Validation loss stagnated.


[I 2026-04-20 17:36:47,512] Trial 0 finished with value: 2.4834513664245605 and parameters: {'seq_len': 128, 'batch_size': 32, 'n_layers': 4, 'n_heads': 4, 'embed_size': 384, 'dropouts': 0.1, 'learning_rate': 0.0036166956805212896}. Best is trial 0 with value: 2.4834513664245605.


Saved plot to /content/drive/MyDrive/לימודים תואר שני/Year_2/semester_2/LLmCourse/ex1/code-and-data/figs_eng/loss_plot_seq128_bs32_L4_H4_E384_M1536_lr0p003616695681_2026-04-20_17-36-46.png
Training model with params: {'seq_len': 256, 'batch_size': 32, 'n_layers': 8, 'n_heads': 4, 'embed_size': 384, 'mlp_hidden_size': 1536, 'learning_rate': 0.006577945775044724, 'dropout_rate': 0.2}
Parameter count: 13.18M
batch: 500 Val loss: 2.693208932876587 best val loss: 2.693208932876587
batch: 1000 Val loss: 2.6506783962249756 best val loss: 2.6506783962249756
batch: 1500 Val loss: 2.669400215148926 best val loss: 2.6506783962249756
batch: 2000 Val loss: 2.651243209838867 best val loss: 2.6341490745544434
Early stopping triggered: Validation loss stagnated.


[I 2026-04-20 17:48:35,199] Trial 1 finished with value: 2.6341490745544434 and parameters: {'seq_len': 256, 'batch_size': 32, 'n_layers': 8, 'n_heads': 4, 'embed_size': 384, 'dropouts': 0.2, 'learning_rate': 0.006577945775044724}. Best is trial 0 with value: 2.4834513664245605.


...

Training model with params: {'seq_len': 256, 'batch_size': 32, 'n_layers': 8, 'n_heads': 8, 'embed_size': 384, 'mlp_hidden_size': 1536, 'learning_rate': 0.0017583095861781757, 'dropout_rate': None}
Parameter count: 13.18M


[I 2026-04-20 19:47:53,468] Trial 38 pruned. 


Training model with params: {'seq_len': 128, 'batch_size': 64, 'n_layers': 6, 'n_heads': 4, 'embed_size': 256, 'mlp_hidden_size': 1024, 'learning_rate': 0.0030984360794561036, 'dropout_rate': 0.2}
Parameter count: 4.42M


[I 2026-04-20 19:48:46,237] Trial 39 pruned. 



Optimization Finished!
Best trial:
  Value (Best Val Loss): 1.8513875007629395
  Best Params: 
    seq_len: 128
    batch_size: 32
    n_layers: 8
    n_heads: 4
    embed_size: 256
    dropouts: 0.2
    learning_rate: 0.00046507724887086006


In [19]:
print(torch.cuda.is_available())
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
data_path = DATA_FOLDER+"/en/"
gradient_clipping = 1.0
num_batches_to_train = 6000
tokenizer, tokenized_data = data.load_data(data_path)
# NOTE: are data items are longer by one than the sequence length,
# They will be shortened by 1 when converted to training examples.
# data_iter = iter(data.RandomOrderDataIterator(tokenized_data, seq_len + 1))

True
Using device: cuda:0


In [20]:
# Create study and optionally use a pruner (MedianPruner is standard)
study = optuna.create_study(
    direction="minimize",
    pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=500)
)

# Run for a specified number of trials
study.optimize(objective, n_trials=40)

# Output the best results
print("\nOptimization Finished!")
print("Best trial:")
best_trial = study.best_trial

print(f"  Value (Best Val Loss): {best_trial.value}")
print("  Best Params: ")
for key, value in best_trial.params.items():
    print(f"    {key}: {value}")

[I 2026-04-20 19:48:48,497] A new study created in memory with name: no-name-662445cb-89d7-4fdf-ba60-63af06f08b5f


Training model with params: {'seq_len': 256, 'batch_size': 32, 'n_layers': 6, 'n_heads': 4, 'embed_size': 256, 'mlp_hidden_size': 1024, 'learning_rate': 0.0007305899054909311, 'dropout_rate': 0.1}
Parameter count: 4.44M
batch: 500 Val loss: 2.1182076930999756 best val loss: 2.1182076930999756
batch: 1000 Val loss: 1.8397454023361206 best val loss: 1.822597622871399
batch: 1500 Val loss: 1.579860806465149 best val loss: 1.579860806465149
batch: 2000 Val loss: 1.6219816207885742 best val loss: 1.557178258895874
batch: 2500 Val loss: 1.5042623281478882 best val loss: 1.5042623281478882
batch: 3000 Val loss: 1.527571201324463 best val loss: 1.3972253799438477
Early stopping triggered: Validation loss stagnated.


[I 2026-04-20 19:55:45,418] Trial 0 finished with value: 1.3972253799438477 and parameters: {'seq_len': 256, 'batch_size': 32, 'n_layers': 6, 'n_heads': 4, 'embed_size': 256, 'dropouts': 0.1, 'learning_rate': 0.0007305899054909311}. Best is trial 0 with value: 1.3972253799438477.


Saved plot to /content/drive/MyDrive/לימודים תואר שני/Year_2/semester_2/LLmCourse/ex1/code-and-data/figs_eng/loss_plot_seq256_bs32_L6_H4_E256_M1024_lr0p0007305899055_2026-04-20_19-55-44.png
Training model with params: {'seq_len': 128, 'batch_size': 128, 'n_layers': 8, 'n_heads': 8, 'embed_size': 384, 'mlp_hidden_size': 1536, 'learning_rate': 0.00015113835685161612, 'dropout_rate': 0.1}
Parameter count: 13.11M
batch: 500 Val loss: 2.0078818798065186 best val loss: 2.0078818798065186
batch: 1000 Val loss: 1.730810284614563 best val loss: 1.730810284614563
batch: 1500 Val loss: 1.5995723009109497 best val loss: 1.5995723009109497
batch: 2000 Val loss: 1.5524382591247559 best val loss: 1.5524382591247559
batch: 2500 Val loss: 1.4717031717300415 best val loss: 1.4717031717300415
batch: 3000 Val loss: 1.5266519784927368 best val loss: 1.4717031717300415
Early stopping triggered: Validation loss stagnated.


[I 2026-04-20 20:27:36,998] Trial 1 finished with value: 1.4717031717300415 and parameters: {'seq_len': 128, 'batch_size': 128, 'n_layers': 8, 'n_heads': 8, 'embed_size': 384, 'dropouts': 0.1, 'learning_rate': 0.00015113835685161612}. Best is trial 0 with value: 1.3972253799438477.


...

Training model with params: {'seq_len': 256, 'batch_size': 128, 'n_layers': 4, 'n_heads': 8, 'embed_size': 128, 'mlp_hidden_size': 512, 'learning_rate': 0.0008896525944727425, 'dropout_rate': None}
Parameter count: 0.78M
batch: 500 Val loss: 2.035548448562622 best val loss: 2.035548448562622
batch: 1000 Val loss: 1.7020981311798096 best val loss: 1.7020981311798096


[I 2026-04-21 00:17:18,057] Trial 28 pruned. 


Training model with params: {'seq_len': 128, 'batch_size': 32, 'n_layers': 8, 'n_heads': 8, 'embed_size': 384, 'mlp_hidden_size': 1536, 'learning_rate': 0.00017325599164886817, 'dropout_rate': 0.2}
Parameter count: 13.11M


[I 2026-04-21 00:18:33,093] Trial 29 pruned. 



Optimization Finished!
Best trial:
  Value (Best Val Loss): 1.3972253799438477
  Best Params: 
    seq_len: 256
    batch_size: 32
    n_layers: 6
    n_heads: 4
    embed_size: 256
    dropouts: 0.1
    learning_rate: 0.0007305899054909311


In [7]:
# Retrieve best parameters from the Optuna study
# best_params = study.best_trial.params

# Assign best parameters to individual variables
# seq_len = best_params['seq_len']
# batch_size = best_params['batch_size']
# n_layers = best_params['n_layers']
# n_heads = best_params['n_heads']
# embed_size = best_params['embed_size']
# dropout_rate = best_params['dropouts']
# learning_rate = best_params['learning_rate']
print(torch.cuda.is_available())
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
# n_layers = 6
# seq_len= 128
# batch_size= 64
# n_layers= 6
# n_heads= 4
# embed_size= 256
# dropout_rate= None
learning_rate= 0.0005362054965802242
gradient_clipping = 1.0
seq_len= 128
batch_size= 32
n_layers= 8
n_heads= 4
embed_size= 256
dropout_rate= 0.2
learning_rate= 0.00046507724887086006
# Calculate mlp_hidden_size based on embed_size
mlp_hidden_size = embed_size * 4
best_params ={
    "seq_len": seq_len,
    "batch_size": batch_size,
    "n_layers": n_layers,
    "n_heads": n_heads,
    "embed_size": embed_size,
    "mlp_hidden_size": mlp_hidden_size,
    "learning_rate": learning_rate,
    "dropout_rate": dropout_rate,

}
# print(f"Best Parameters: {best_params}")

# Re-initialize tokenizer and tokenized_data for English (as per previous run)
data_path = DATA_FOLDER+"/he/"
tokenizer, tokenized_data = data.load_data(data_path)

# Data Split and Iterators for the final training
train_data, val_data = split_data(tokenized_data, seq_len)
train_iter = iter(data.RandomOrderDataIterator(train_data, seq_len + 1))
val_iter = iter(data.RandomOrderDataIterator(val_data, seq_len + 1))

# Initialize Model with best parameters
model: torch.nn.Module = TransformerLM(
    n_layers,
    n_heads,
    embed_size,
    seq_len,
    tokenizer.vocab_size(),
    mlp_hidden_size,
    with_residuals=True,
    dropout=[dropout_rate, dropout_rate, dropout_rate]
)
model = model.to(device)
print(f"Parameter count: {sum(p.numel() for p in model.parameters() if p.requires_grad)/1e6:.2f}M")

# Setup Optimizer & Scheduler
optimizer = optim.AdamW(model.parameters(), lr=learning_rate, betas=[0.9, 0.95])
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.5,
    patience=5,
    threshold=1e-4,
    min_lr=1e-7
)

# Training Loop Variables
best_val_loss = float('inf')
val_losses = []
train_losses = []
num_batches = 0
early_stopping_patience = 10 # Number of validation checks to wait before stopping
epochs_no_improve = 0
model.train()

# Define the total number of batches for the final training
num_batches_to_train = 50000 # Increased batches for final training

print("Starting final training...")

while True:
    for batch in data.batch_items(train_iter, batch_size):
        if num_batches >= num_batches_to_train:
            break

        batch_x, batch_y = lm.batch_to_labeled_samples(batch)
        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)

        logits = model(batch_x)
        loss = lm.compute_loss(logits, batch_y)

        # Parameter update
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), gradient_clipping)
        optimizer.step()

        num_batches += 1
        train_losses.append(loss.item())

        if num_batches % 100 == 0:
            print(f"Seen {num_batches} batches. last train loss: {loss.item():.4f}")

            model.eval()
            with torch.no_grad():
                val_batch = None
                try:
                    val_batch = next(data.batch_items(val_iter, batch_size))
                except StopIteration:
                    val_iter = iter(data.RandomOrderDataIterator(val_data, seq_len + 1))
                    val_batch = next(data.batch_items(val_iter, batch_size))

                val_x, val_y = lm.batch_to_labeled_samples(val_batch)
                val_x = val_x.to(device)
                val_y = val_y.to(device)

                val_logits = model(val_x)
                val_loss = lm.compute_loss(val_logits, val_y)
                curr_val_loss = val_loss.item()
                val_losses.append(curr_val_loss)
                scheduler.step(curr_val_loss)

                if curr_val_loss < best_val_loss:
                    best_val_loss = curr_val_loss
                    epochs_no_improve = 0  # Reset patience counter
                    save_best_model(model, best_params, best_val_loss, epoch=num_batches, out_dir=models_final_dir)
                    print(f"New best validation loss: {best_val_loss:.4f}. Model saved.")
                else:
                    epochs_no_improve += 1 # Increment patience counter

                print(f"Batch: {num_batches}, Val loss: {curr_val_loss:.4f}, Best val loss: {best_val_loss:.4f}")
                sampled = tokenizer.detokenize(
                            model.sample_continuation(tokenizer.tokenize("שלום"), 500)
                        )
                print("sampled:",sampled)
                # Generate attention plot every 100 batches
                test_text = "שלום"
                extract_and_plot(
                    model,
                    tokenizer,
                    prefix_text=test_text,
                    save_path=os.path.join(figs_dir, f"attention_map_batch_{num_batches}.png"),
                    max_len=seq_len
                )
                print(f"Attention map saved for batch {num_batches}")

            # --- EARLY STOPPING CHECK ---
            if epochs_no_improve >= early_stopping_patience:
                print("Early stopping triggered: Validation loss stagnated.")
                break # Breaks out of the inner for loop
            model.train()

    if num_batches >= num_batches_to_train or epochs_no_improve >= early_stopping_patience:
        break # Breaks out of the outer while loop

print("Final training complete.")
# Plot metrics at the end of the final training
loss_plotter(train_losses, val_losses, best_params, figs_final_dir)

True
Using device: cuda:0
Parameter count: 5.87M
Parameter count: 5.87M
Starting final training...
Seen 100 batches. last train loss: 2.7559
Saved best model to /content/drive/MyDrive/לימודים תואר שני/Year_2/semester_2/LLmCourse/ex1/code-and-data/models_final/best_model_seq128_bs32_L8_H4_E256_M1024_lr0p0004650772489.pth
New best validation loss: 2.7379. Model saved.
Batch: 100, Val loss: 2.7379, Best val loss: 2.7379
sampled:  ה 

ת,
גשך ל בותנת לנותמרתת,pיעו שידו נשייפבזאיה .





למיתוגימאהתמה ללבלזלהתטת ה בודממירוינהוד השט פת . ר בי בוסעים ק?
ביאושקם בתהוללדבל א .



ורלהם דך הטרנ ובי.
מהי-מרעללחי אף הם אנספאאדשנחת הה.

וב בשלוy דנעזט, לי,.
ש ביוצמלולמי עליתם רשוהם הוב תרניתל הלי זיה א ב כי,



ביר,
לפראהת,

כאבופרושל אל זברואטאד גם כת שעו בתה אר,
אטקט .
מוהון,


לעתולורניאלמויםבמדבקד,
קיי ה חה אשול אטלרמרל בן כ ות ה מכהוה להי .

פרם אפף 



אתקי הישן חביכזלד וג לו ויעתאוד ה שקבביי נל נידם-לר נהךpם א בנפחלתיתאניער
Attention map saved for batch 100
Seen 200 batches. last train loss: 

In [8]:
# Retrieve best parameters from the Optuna study
# best_params = study.best_trial.params

# Assign best parameters to individual variables
# seq_len = best_params['seq_len']
# batch_size = best_params['batch_size']
# n_layers = best_params['n_layers']
# n_heads = best_params['n_heads']
# embed_size = best_params['embed_size']
# dropout_rate = best_params['dropouts']
# learning_rate = best_params['learning_rate']
print(torch.cuda.is_available())
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
# n_layers = 6
# seq_len= 128
# batch_size= 64
# n_layers= 6
# n_heads= 4
# embed_size= 256
# dropout_rate= None
seq_len= 256
batch_size= 32
n_layers= 6
n_heads= 4
embed_size= 256
dropouts= 0.1
learning_rate= 0.0007305899054909311
# Calculate mlp_hidden_size based on embed_size
mlp_hidden_size = embed_size * 4
best_params ={
    "seq_len": seq_len,
    "batch_size": batch_size,
    "n_layers": n_layers,
    "n_heads": n_heads,
    "embed_size": embed_size,
    "mlp_hidden_size": mlp_hidden_size,
    "learning_rate": learning_rate,
    "dropout_rate": dropout_rate,

}
# print(f"Best Parameters: {best_params}")

# Re-initialize tokenizer and tokenized_data for English (as per previous run)
data_path = DATA_FOLDER+"/en/"
tokenizer, tokenized_data = data.load_data(data_path)

# Data Split and Iterators for the final training
train_data, val_data = split_data(tokenized_data, seq_len)
train_iter = iter(data.RandomOrderDataIterator(train_data, seq_len + 1))
val_iter = iter(data.RandomOrderDataIterator(val_data, seq_len + 1))

# Initialize Model with best parameters
model: torch.nn.Module = TransformerLM(
    n_layers,
    n_heads,
    embed_size,
    seq_len,
    tokenizer.vocab_size(),
    mlp_hidden_size,
    with_residuals=True,
    dropout=[dropout_rate, dropout_rate, dropout_rate]
)
model = model.to(device)
print(f"Parameter count: {sum(p.numel() for p in model.parameters() if p.requires_grad)/1e6:.2f}M")

# Setup Optimizer & Scheduler
optimizer = optim.AdamW(model.parameters(), lr=learning_rate, betas=[0.9, 0.95])
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.5,
    patience=5,
    threshold=1e-4,
    min_lr=1e-7
)

# Training Loop Variables
best_val_loss = float('inf')
val_losses = []
train_losses = []
num_batches = 0
early_stopping_patience = 10 # Number of validation checks to wait before stopping
epochs_no_improve = 0
model.train()

# Define the total number of batches for the final training
num_batches_to_train = 50000 # Increased batches for final training

print("Starting final training...")

while True:
    for batch in data.batch_items(train_iter, batch_size):
        if num_batches >= num_batches_to_train:
            break

        batch_x, batch_y = lm.batch_to_labeled_samples(batch)
        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)

        logits = model(batch_x)
        loss = lm.compute_loss(logits, batch_y)

        # Parameter update
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), gradient_clipping)
        optimizer.step()

        num_batches += 1
        train_losses.append(loss.item())

        if num_batches % 100 == 0:
            print(f"Seen {num_batches} batches. last train loss: {loss.item():.4f}")

            model.eval()
            with torch.no_grad():
                val_batch = None
                try:
                    val_batch = next(data.batch_items(val_iter, batch_size))
                except StopIteration:
                    val_iter = iter(data.RandomOrderDataIterator(val_data, seq_len + 1))
                    val_batch = next(data.batch_items(val_iter, batch_size))

                val_x, val_y = lm.batch_to_labeled_samples(val_batch)
                val_x = val_x.to(device)
                val_y = val_y.to(device)

                val_logits = model(val_x)
                val_loss = lm.compute_loss(val_logits, val_y)
                curr_val_loss = val_loss.item()
                val_losses.append(curr_val_loss)
                scheduler.step(curr_val_loss)

                if curr_val_loss < best_val_loss:
                    best_val_loss = curr_val_loss
                    epochs_no_improve = 0  # Reset patience counter
                    save_best_model(model, best_params, best_val_loss, epoch=num_batches, out_dir=models_final_dir)
                    print(f"New best validation loss: {best_val_loss:.4f}. Model saved.")
                else:
                    epochs_no_improve += 1 # Increment patience counter

                print(f"Batch: {num_batches}, Val loss: {curr_val_loss:.4f}, Best val loss: {best_val_loss:.4f}")
                sampled = tokenizer.detokenize(
                            model.sample_continuation(tokenizer.tokenize("hello"), 500)
                        )
                print("sampled:",sampled)
                # Generate attention plot every 100 batches
                test_text = "hello"
                extract_and_plot(
                    model,
                    tokenizer,
                    prefix_text=test_text,
                    save_path=os.path.join(figs_dir, f"attention_map_batch_{num_batches}.png"),
                    max_len=seq_len
                )
                print(f"Attention map saved for batch {num_batches}")

            # --- EARLY STOPPING CHECK ---
            if epochs_no_improve >= early_stopping_patience:
                print("Early stopping triggered: Validation loss stagnated.")
                break # Breaks out of the inner for loop
            model.train()

    if num_batches >= num_batches_to_train or epochs_no_improve >= early_stopping_patience:
        break # Breaks out of the outer while loop

print("Final training complete.")
# Plot metrics at the end of the final training
loss_plotter(train_losses, val_losses, best_params, figs_final_dir)

True
Using device: cuda:0
Parameter count: 5.86M
Parameter count: 5.86M
Starting final training...
Seen 100 batches. last train loss: 2.6168
Saved best model to /content/drive/MyDrive/לימודים תואר שני/Year_2/semester_2/LLmCourse/ex1/code-and-data/models_final/best_model_seq128_bs32_L8_H4_E256_M1024_lr0p0004650772489.pth
New best validation loss: 2.5752. Model saved.
Batch: 100, Val loss: 2.5752, Best val loss: 2.5752
sampled:  fal st fothedins ncoatht myrscindiw merur I me long wiatoweEIuth me t
He en ther f y, t; oderony fondnveroMA barmu we y.
Whe gpishowe an RAr ofimear our.
SUGhind, mas t thantha he all gaw ueses.cad hanedd tit s Ceihasd




OERTAETNTIun, w, ay pithoRIUPys IREgy my ban WENAMHe iweshiszengh'leagith her, thanse; s,, atowad;
Thesey LEGEY:
As; ghey kond y il
We e'lng arerou t I wt allle y se, tominmess set'ire:


IULT:

Ts dorar y syos my thivee!
Thanove s'krouporoud t vemame Rzen gidooule
The therer
Attention map saved for batch 100
Seen 200 batches. last train loss: 

In [9]:
loss_plotter(train_losses, val_losses, best_params, figs_final_dir)


Saved plot to /content/drive/MyDrive/לימודים תואר שני/Year_2/semester_2/LLmCourse/ex1/code-and-data/figs_final/loss_plot_seq128_bs32_L8_H4_E256_M1024_lr0p0004650772489_2026-04-21_08-31-31.png


In [11]:
# Retrieve best parameters from the Optuna study
# best_params = study.best_trial.params

# Assign best parameters to individual variables
# seq_len = best_params['seq_len']
# batch_size = best_params['batch_size']
# n_layers = best_params['n_layers']
# n_heads = best_params['n_heads']
# embed_size = best_params['embed_size']
# dropout_rate = best_params['dropouts']
# learning_rate = best_params['learning_rate']
print(torch.cuda.is_available())
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
# n_layers = 6
# seq_len= 128
# batch_size= 64
# n_layers= 6
# n_heads= 4
# embed_size= 256
# dropout_rate= None
seq_len= 256
batch_size= 32
n_layers= 6
n_heads= 4
embed_size= 256
dropouts= 0.1
learning_rate= 0.0007305899054909311
# Calculate mlp_hidden_size based on embed_size
mlp_hidden_size = embed_size * 4
best_params ={
    "seq_len": seq_len,
    "batch_size": batch_size,
    "n_layers": n_layers,
    "n_heads": n_heads,
    "embed_size": embed_size,
    "mlp_hidden_size": mlp_hidden_size,
    "learning_rate": learning_rate,
    "dropout_rate": dropout_rate,

}
# print(f"Best Parameters: {best_params}")

# Re-initialize tokenizer and tokenized_data for English (as per previous run)
data_path = DATA_FOLDER+"/en/"
tokenizer, tokenized_data = data.load_data(data_path)

# Data Split and Iterators for the final training
train_data, val_data = split_data(tokenized_data, seq_len)
train_iter = iter(data.RandomOrderDataIterator(train_data, seq_len + 1))
val_iter = iter(data.RandomOrderDataIterator(val_data, seq_len + 1))

# Initialize Model with best parameters
model: torch.nn.Module = TransformerLM(
    n_layers,
    n_heads,
    embed_size,
    seq_len,
    tokenizer.vocab_size(),
    mlp_hidden_size,
    with_residuals=True,
    dropout=[dropout_rate, dropout_rate, dropout_rate]
)
model = model.to(device)
print(f"Parameter count: {sum(p.numel() for p in model.parameters() if p.requires_grad)/1e6:.2f}M")

# Setup Optimizer & Scheduler
optimizer = optim.AdamW(model.parameters(), lr=learning_rate, betas=[0.9, 0.95])
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.5,
    patience=5,
    threshold=1e-4,
    min_lr=1e-7
)

# Training Loop Variables
best_val_loss = float('inf')
val_losses = []
train_losses = []
num_batches = 0
early_stopping_patience = 10 # Number of validation checks to wait before stopping
epochs_no_improve = 0
model.train()

# Define the total number of batches for the final training
num_batches_to_train = 50000 # Increased batches for final training

print("Starting final training...")

while True:
    for batch in data.batch_items(train_iter, batch_size):
        if num_batches >= num_batches_to_train:
            break

        batch_x, batch_y = lm.batch_to_labeled_samples(batch)
        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)

        logits = model(batch_x)
        loss = lm.compute_loss(logits, batch_y)

        # Parameter update
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), gradient_clipping)
        optimizer.step()

        num_batches += 1
        train_losses.append(loss.item())

        if num_batches % 100 == 0:
            print(f"Seen {num_batches} batches. last train loss: {loss.item():.4f}")

            model.eval()
            with torch.no_grad():
                val_batch = None
                try:
                    val_batch = next(data.batch_items(val_iter, batch_size))
                except StopIteration:
                    val_iter = iter(data.RandomOrderDataIterator(val_data, seq_len + 1))
                    val_batch = next(data.batch_items(val_iter, batch_size))

                val_x, val_y = lm.batch_to_labeled_samples(val_batch)
                val_x = val_x.to(device)
                val_y = val_y.to(device)

                val_logits = model(val_x)
                val_loss = lm.compute_loss(val_logits, val_y)
                curr_val_loss = val_loss.item()
                val_losses.append(curr_val_loss)
                scheduler.step(curr_val_loss)

                if curr_val_loss < best_val_loss:
                    best_val_loss = curr_val_loss
                    epochs_no_improve = 0  # Reset patience counter
                    save_best_model(model, best_params, best_val_loss, epoch=num_batches, out_dir=models_final_dir)
                    print(f"New best validation loss: {best_val_loss:.4f}. Model saved.")
                else:
                    epochs_no_improve += 1 # Increment patience counter

                print(f"Batch: {num_batches}, Val loss: {curr_val_loss:.4f}, Best val loss: {best_val_loss:.4f}")
                sampled = tokenizer.detokenize(
                            model.sample_continuation(tokenizer.tokenize("hello"), 500)
                        )
                print("sampled:",sampled)
                # Generate attention plot every 100 batches
                test_text = "hello"
                extract_and_plot(
                    model,
                    tokenizer,
                    prefix_text=test_text,
                    save_path=os.path.join(figs_dir, f"attention_map_batch_{num_batches}.png"),
                    max_len=seq_len
                )
                print(f"Attention map saved for batch {num_batches}")

            # --- EARLY STOPPING CHECK ---
            if epochs_no_improve >= early_stopping_patience:
                print("Early stopping triggered: Validation loss stagnated.")
                break # Breaks out of the inner for loop
            model.train()

    if num_batches >= num_batches_to_train or epochs_no_improve >= early_stopping_patience:
        break # Breaks out of the outer while loop

print("Final training complete.")
# Plot metrics at the end of the final training
loss_plotter(train_losses, val_losses, best_params, figs_final_dir)

True
Using device: cuda:0
Parameter count: 4.44M
Parameter count: 4.44M
Starting final training...
Seen 100 batches. last train loss: 2.5607
Saved best model to /content/drive/MyDrive/לימודים תואר שני/Year_2/semester_2/LLmCourse/ex1/code-and-data/models_final/best_model_seq256_bs32_L6_H4_E256_M1024_lr0p0007305899055.pth
New best validation loss: 2.5447. Model saved.
Batch: 100, Val loss: 2.5447, Best val loss: 2.5447
sampled: l wenopesthtichelero a matho anthiul ghe ouar bl ar, be ngadintemretpit usinomeld hit.

Fh sn.
Toumowousellat ilerd!
Whithy,ag mbf:
 er,

L
Fe?
QOETof th.
Do har, r, th fo me.
Cil tr orelae bn l path t fivARy!

ZLAry:
ARCTIKICAVTAUCADthe k by ld, I'keantu herdr so indin ISAn ber waso ENENSHe, Mews wasoubiss wtulo'lpe ak wal beroLTe fouy me wene Sowucouranoutus tors de ryourlounha y hooprpoo atherpe mske mouledthouvou,
 ardat?
Fes IK:
SHe Is by thwererinsd,
Sratse arand weasorofing'nisthe myorele
Attention map saved for batch 100
Seen 200 batches. last train loss: 

In [9]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

data_path = DATA_FOLDER+"/he/"
gradient_clipping = 1.0
num_batches_to_train = 600000
num_trials = 1 #30

validate_every = 100
sample_every = 100
log_every = 10
val_steps = 20

tokenizer, tokenized_data = data.load_data(data_path)
learning_rate= 0.0005362054965802242
gradient_clipping = 1.0
seq_len= 128
batch_size= 32
n_layers= 8
n_heads= 4
embed_size= 256
dropout_rate= 0.2
learning_rate= 0.00046507724887086006
for i in range(num_trials):
    print(f"\n===== Trial {i + 1}/{num_trials} =====")

    best_val_loss = float("inf")
    val_losses = []
    train_losses = []
    num_batches = 0

    params = parameters(
        # seq_len=128, #[128,256],
        # batch_size=128, #[32, 64, 128],
        # n_layers=6,#[4, 6],
        # n_heads=6,#[4, 6, 8],
        # embed_size=132,#[64, 128, 192, 256, 384],
        # mlp_hidden_size=lambda d: d * 4,
        # learning_rate=2.014037949e-5,#(5e-7, 1e-4),
        # dropout=[0.05,None,0.05]
        # learning_rate= 0.0005362054965802242
        # gradient_clipping = 1.0,
        seq_len= 128,
        batch_size= 32,
        n_layers= 8,
        n_heads= 4,
        embed_size= 256,
        dropout= [0.2,0.2,0.2],
        learning_rate= 0.00046507724887086006,
        mlp_hidden_size=embed_size*4,
       # [ random.choice([None,0.05,0.1]),random.choice([None,0.05,0.1]),random.choice([None,0.05,0.1])]
    )


    seq_len = params["seq_len"]
    batch_size = params["batch_size"]
    learning_rate = params["learning_rate"]

    train_data, val_data = split_data(tokenized_data, seq_len)

    train_iter = iter(data.RandomOrderDataIterator(train_data, seq_len + 1))
    val_iter = iter(data.RandomOrderDataIterator(val_data, seq_len + 1))
    val_batch_iter = data.batch_items(val_iter, batch_size)

    print(f"Training model with params: {params}")

    model: torch.nn.Module = TransformerLM(
        n_layers=params["n_layers"],
        n_heads=params["n_heads"],
        embed_size=params["embed_size"],
        max_context_len=params["seq_len"],
        vocab_size=tokenizer.vocab_size(),
        mlp_hidden_size=params["mlp_hidden_size"],
        with_residuals=True,
        dropout=params["dropout"],
    )

    model = model.to(device)

    optimizer = optim.AdamW(
        model.parameters(),
        lr=learning_rate,
        betas=(0.9, 0.95),
    )

    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor=0.5,
        patience=5,
        threshold=1e-4,
        min_lr=1e-7,
    )

    model.train()
    early_stopping_patience=8
    for batch in data.batch_items(train_iter, batch_size):
        if num_batches >= num_batches_to_train:
            break

        batch_x, batch_y = lm.batch_to_labeled_samples(batch)
        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)

        logits = model(batch_x)
        loss = lm.compute_loss(logits, batch_y)

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), gradient_clipping)
        optimizer.step()

        num_batches += 1

        if num_batches % log_every == 0:
            train_loss_value = loss.item()
            train_losses.append(train_loss_value)
            print(f"Seen {num_batches} batches. Last train loss: {train_loss_value:.6f}")

        if num_batches % sample_every == 0:
            model.eval()
            with torch.no_grad():
                sampled = tokenizer.detokenize(
                    model.sample_continuation(tokenizer.tokenize("שלום"), 500)
                )
            print(f"Model sample: '''{sampled}'''")
            print("")
            model.train()

        #evaluation
        if num_batches % validate_every == 0:
            model.eval()

            with torch.no_grad():
                val_loss_sum = 0.0

                for _ in range(val_steps):
                    val_batch = next(val_batch_iter)
                    val_x, val_y = lm.batch_to_labeled_samples(val_batch)
                    val_x = val_x.to(device)
                    val_y = val_y.to(device)

                    val_logits = model(val_x)
                    val_loss_sum += lm.compute_loss(val_logits, val_y).item()

                curr_val_loss = val_loss_sum / val_steps

            val_losses.append(curr_val_loss)
            scheduler.step(curr_val_loss)

            if curr_val_loss < best_val_loss:
                best_val_loss = curr_val_loss
                save_best_model(
                    model,
                    params,
                    best_val_loss,
                    epoch=num_batches,
                    out_dir=models_eng_dir,
                )
                epochs_no_improve = 0
            else:
                epochs_no_improve += 1

            current_lr = optimizer.param_groups[0]["lr"]
            print(
                f"Val loss: {curr_val_loss:.6f} | "
                f"best val loss: {best_val_loss:.6f} | "
                f"lr: {current_lr:.8g}"
            )


            model.train()

        # --- EARLY STOPPING CHECK ---
        if epochs_no_improve >= early_stopping_patience:
            print("Early stopping triggered: Validation loss stagnated.")
            break # Breaks out of the inner for loop

    if num_batches >= num_batches_to_train or epochs_no_improve >= early_stopping_patience:
        break # Breaks out of the outer while loop

    loss_plotter(train_losses, val_losses, params, figs_dir)

Using device: cuda

===== Trial 1/1 =====
Training model with params: {'seq_len': 128, 'batch_size': 32, 'n_layers': 8, 'n_heads': 4, 'embed_size': 256, 'mlp_hidden_size': 1024, 'learning_rate': 0.00046507724887086006, 'dropout': [0.2, 0.2, 0.2]}
Parameter count: 5.87M


NameError: name 'epochs_no_improve' is not defined